# 04 · Clean the weather and PLUTO data and engineer weather features
**Outputs:** `data/processed/weather_clean.parquet` and `data/processed/pluto_clean.parquet`, plus the audit trail in `reports/cleaning_report_weather_pluto.json`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from src import config, preprocess as P

pd.set_option("display.float_format", "{:,.2f}".format)

## Weather
Cleaning steps:
- Rename the columns and check that the daily series is complete; any gaps up to 3 days are interpolated.
- Check that `tmin ≤ tmax`.

New feature columns:
| Feature | Meaning |
|---|---|
| `hdd` | Heating degree days: max(0, 65 − mean temp), a standard measure of heating demand |
| `tmin_lag1`, `tmin_lag2`, `precip_lag1`, `precip_lag2` | The previous one and two days. Complaints can lag the weather. |
| `tmin_3day_mean`, `temp_drop_24h` | Sustained cold, and sudden cold snaps |
| `heavy_rain` (≥ 0.5 in), `snow_day` | Event flags |

In [2]:
weather, rep_w = P.clean_weather()
display(pd.DataFrame(rep_w.steps))
weather.head()

,step,rows_before,rows_after,rows_removed,detail,missing_days
0,validate_and_features,1096,1096,0,0 missing days filled,0


,date,tmax,tmin,tmean,precip,rain,snow,wind_max,hdd,tmin_lag1,precip_lag1,tmin_lag2,precip_lag2,tmin_3day_mean,temp_drop_24h,heavy_rain,snow_day
0,2023-01-01,52.80,40.50,47.90,0.04,0.04,0.00,10.70,17.10,NaN,NaN,NaN,NaN,40.50,NaN,False,False
1,2023-01-02,54.70,40.50,46.60,0.04,0.04,0.00,5.50,18.40,40.50,0.04,NaN,NaN,40.50,0.00,False,False
2,2023-01-03,55.60,44.80,49.60,0.46,0.46,0.00,12.00,15.40,40.50,0.04,40.50,0.04,41.93,-4.30,False,False
3,2023-01-04,64.00,51.00,56.00,0.08,0.08,0.00,10.80,9.00,44.80,0.46,40.50,0.04,45.43,-6.20,False,False
4,2023-01-05,55.60,43.60,49.20,0.00,0.00,0.00,7.90,15.80,51.00,0.08,44.80,0.46,46.47,7.40,False,False


In [3]:
weather[["tmin", "hdd", "temp_drop_24h", "precip", "heavy_rain", "snow_day"]].describe().T

,count,mean,std,min,25%,50%,75%,max
tmin,"1,096.00",46.89,16.13,1.40,33.88,46.90,60.62,83.20
hdd,"1,096.00",12.89,13.26,0.00,0.00,9.45,23.60,54.00
temp_drop_24h,"1,095.00",0.01,5.11,-22.10,-2.90,0.30,3.35,19.60
precip,"1,096.00",0.14,0.34,0.00,0.00,0.00,0.10,3.69


## PLUTO
Normalize the BBL to 10 digits, convert numeric columns, set `yearbuilt = 0` to missing, validate ZIPs and coordinates, and flag residential lots.

In [4]:
pluto, rep_p = P.clean_pluto()
display(pd.DataFrame(rep_p.steps))
pluto[["bbl", "borough", "zipcode", "yearbuilt", "unitsres", "is_residential"]].head()

,step,rows_before,rows_after,rows_removed,detail,unknown_yearbuilt
0,load_raw,858312,858312,0,,NaN
1,valid_unique_bbl,858312,858312,0,invalid or duplicate BBL,NaN
2,convert_types,858312,858312,0,"39,842 lots with yearbuilt=0 set to missing","39,842.00"


,bbl,borough,zipcode,yearbuilt,unitsres,is_residential
0,1000010010,MANHATTAN,10004,"1,900.00",0.00,False
1,1000010100,MANHATTAN,<NA>,NaN,0.00,False
2,1000010101,MANHATTAN,10004,"1,900.00",0.00,False
3,1000010111,MANHATTAN,10004,NaN,0.00,False
4,1000010112,MANHATTAN,10004,NaN,0.00,False


In [5]:
# Validation checks
assert pluto["bbl"].str.fullmatch(r"[1-5]\d{9}").all(), "bad BBL"
assert pluto["bbl"].is_unique, "duplicate BBL"
assert (weather["tmin"] <= weather["tmax"]).all()
print("All checks passed")
pluto.groupby("borough")[["unitsres"]].sum().assign(median_year_built=pluto.groupby("borough")["yearbuilt"].median())

All checks passed


,unitsres,median_year_built
borough,,
BRONX,"591,521.00","1,935.00"
BROOKLYN,"1,117,462.00","1,925.00"
MANHATTAN,"960,487.00","1,910.00"
QUEENS,"891,982.00","1,935.00"
STATEN ISLAND,"180,530.00","1,970.00"


## Save

In [6]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
weather.to_parquet(config.CLEAN_WEATHER_FILE, index=False)
pluto.to_parquet(config.CLEAN_PLUTO_FILE, index=False)
(config.REPORTS_DIR / "cleaning_report_weather_pluto.json").write_text(
    json.dumps([rep_w.to_dict(), rep_p.to_dict()], indent=2, default=str))
print("Saved:", config.CLEAN_WEATHER_FILE.name, config.CLEAN_PLUTO_FILE.name)

Saved: weather_clean.parquet pluto_clean.parquet


→ Next: **05_build_project.ipynb**